In [1]:
# ============================================================
# CELL 1: SETUP
# ============================================================
import os
import gc
import numpy as np
import pandas as pd
import torch

# Working directory ঠিক করুন
PROJECT_DIR = r"C:\Users\User\Desktop\research\bengali-smishing"
os.chdir(PROJECT_DIR)
print(f"✅ Working directory: {os.getcwd()}")

# Device check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")

# Data file check
print(f"\n📁 File check:")
print(f"  train_hard2.csv: {os.path.exists('data/processed/train_hard2.csv')}")
print(f"  test_hard2.csv:  {os.path.exists('data/processed/test_hard2.csv')}")
print(f"  lora_b1_final:   {os.path.exists('results/models/lora_b1_final')}")

✅ Working directory: C:\Users\User\Desktop\research\bengali-smishing
✅ Device: cuda

📁 File check:
  train_hard2.csv: True
  test_hard2.csv:  True
  lora_b1_final:   True


In [2]:
# ============================================================
# CELL 2: STATISTICAL VERIFICATION
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Load data
print("Loading data...")
train_df = pd.read_csv("data/processed/train_hard2.csv")
val_df = pd.read_csv("data/processed/test_hard2.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# ---------- Word-level similarity ----------
print("\nComputing word-level similarity...")
vec_w = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_w = vec_w.fit_transform(train_df['text_clean'].astype(str))
X_val_w = vec_w.transform(val_df['text_clean'].astype(str))

sim_w = cosine_similarity(X_val_w, X_train_w)
max_w = sim_w.max()
mean_w = sim_w.mean()

print(f"Word-level Max:  {max_w:.4f}")
print(f"Word-level Mean: {mean_w:.4f}")

# ---------- Character-level similarity ----------
print("\nComputing character-level similarity...")
vec_c = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True
)
X_train_c = vec_c.fit_transform(train_df['text_clean'].astype(str))
X_val_c = vec_c.transform(val_df['text_clean'].astype(str))

sim_c = cosine_similarity(X_val_c, X_train_c)
max_c = sim_c.max()
mean_c = sim_c.mean()

print(f"Char-level Max:  {max_c:.4f}")
print(f"Char-level Mean: {mean_c:.4f}")

# ---------- Summary ----------
print("\n" + "=" * 50)
print("VERIFICATION SUMMARY")
print("=" * 50)
print(f"Word-level Max Similarity: {max_w:.4f}  (threshold: 0.55)")
print(f"Char-level Max Similarity: {max_c:.4f}  (threshold: 0.80)")
print("=" * 50)

if max_w < 0.55 and max_c < 0.80:
    print("✅ All samples below thresholds — clean split")
else:
    print("⚠️ Some samples exceed thresholds — check")

Loading data...
Train: 4898 | Val: 2107

Computing word-level similarity...
Word-level Max:  1.0000
Word-level Mean: 0.0090

Computing character-level similarity...
Char-level Max:  1.0000
Char-level Mean: 0.0171

VERIFICATION SUMMARY
Word-level Max Similarity: 1.0000  (threshold: 0.55)
Char-level Max Similarity: 1.0000  (threshold: 0.80)
⚠️ Some samples exceed thresholds — check


In [3]:
# ============================================================
# CELL 3: BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import PeftModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ---------- Load model ----------
print("Loading model...")
model_path = "results/models/lora_b1_final"
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

base_model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base", num_labels=3
)
model = PeftModel.from_pretrained(base_model, model_path)
model.to(device)
model.eval()
print("✅ Model loaded")

# ---------- Dataset ----------
class SMSDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len
        )
        return enc

labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}

texts = val_df['text_clean'].astype(str).tolist()
y_true = np.array([label2id[l] for l in val_df['label']])

ds = SMSDataset(texts, tokenizer)
loader = DataLoader(ds, batch_size=32, collate_fn=DataCollatorWithPadding(tokenizer))

# ---------- Predictions ----------
print("Computing predictions...")
all_preds = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = torch.argmax(model(**batch).logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())

y_pred = np.array(all_preds)
print("✅ Predictions complete")

# ---------- Bootstrap ----------
def bootstrap_ci(y_true, y_pred, metric_fn, n_iterations=1000):
    scores = []
    np.random.seed(42)
    for _ in range(n_iterations):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)
    return np.mean(scores), lower, upper

print("\n" + "=" * 60)
print("BOOTSTRAP 95% CONFIDENCE INTERVALS (1000 iterations)")
print("=" * 60)

# Macro F1
mean, lo, hi = bootstrap_ci(y_true, y_pred, 
    lambda t, p: f1_score(t, p, average='macro'))
print(f"Macro F1:        {mean:.4f} [{lo:.4f}, {hi:.4f}]")

# Smish Recall
smish_idx = label2id['smish']
mean, lo, hi = bootstrap_ci(y_true, y_pred,
    lambda t, p: (p[t==smish_idx]==smish_idx).mean())
print(f"Smish Recall:    {mean:.4f} [{lo:.4f}, {hi:.4f}]")

# Smish Precision
mean, lo, hi = bootstrap_ci(y_true, y_pred,
    lambda t, p: (t[p==smish_idx]==smish_idx).mean() if (p==smish_idx).sum()>0 else 0)
print(f"Smish Precision: {mean:.4f} [{lo:.4f}, {hi:.4f}]")

# Accuracy
mean, lo, hi = bootstrap_ci(y_true, y_pred, accuracy_score)
print(f"Accuracy:        {mean:.4f} [{lo:.4f}, {hi:.4f}]")

print("=" * 60)

Loading model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded
Computing predictions...
✅ Predictions complete

BOOTSTRAP 95% CONFIDENCE INTERVALS (1000 iterations)
Macro F1:        0.7466 [0.7281, 0.7652]
Smish Recall:    0.5891 [0.5608, 0.6168]
Smish Precision: 0.9866 [0.9779, 0.9943]
Accuracy:        0.7455 [0.7257, 0.7641]


In [4]:
# ============================================================
# CELL 4: CHECK EXISTING AUGMENTATION ABLATION
# ============================================================
import json

print("Checking for existing ablation results...")

# Check various possible files
possible_files = [
    "results/analysis/ablation_augmentation.json",
    "results/analysis/no_aug_results.json",
    "results/no_aug_results.json",
    "results/ablation_augmentation.json",
]

found = False
for f in possible_files:
    if os.path.exists(f):
        print(f"\n✅ Found: {f}")
        with open(f, "r") as file:
            data = json.load(file)
        print(json.dumps(data, indent=2))
        found = True

if not found:
    print("\n⚠️ No existing augmentation ablation found")
    print("\nAvailable files in results/analysis/:")
    if os.path.exists("results/analysis"):
        for f in os.listdir("results/analysis"):
            print(f"  - {f}")
    
    print("\nAvailable files in results/:")
    if os.path.exists("results"):
        for f in os.listdir("results"):
            print(f"  - {f}")

Checking for existing ablation results...

⚠️ No existing augmentation ablation found

Available files in results/analysis/:
  - cross_dataset_bangalabarta.json
  - cv_results.json
  - error_summary.json
  - misclassified_smish.csv
  - random_vs_unseen.json
  - robustness_results.json
  - statistical_test.json

Available files in results/:
  - ablation_lora_r.json
  - all_baselines_final.json
  - analysis
  - baseline_results.json
  - complete_experimental_results.json
  - figures
  - final_main_results.json
  - final_results_b1.json
  - final_results_b2.json
  - hard_split_results.json
  - lora_b1_dedup_results.json
  - lora_campaign_final.json
  - models
  - muril_comparison.json


In [5]:
# ============================================================
# CELL 2b: EXACT DUPLICATE CHECK
# ============================================================
import pandas as pd

train_df = pd.read_csv("data/processed/train_hard2.csv")
test_df = pd.read_csv("data/processed/test_hard2.csv")

train_texts = set(train_df['text_clean'].astype(str))
test_texts = set(test_df['text_clean'].astype(str))

overlap = train_texts & test_texts
print(f"Train unique: {len(train_texts)}")
print(f"Test unique:  {len(test_texts)}")
print(f"Overlap:      {len(overlap)}")

if overlap:
    print(f"\n⚠️ {len(overlap)} exact duplicates found!")
    print("Sample overlaps:")
    for t in list(overlap)[:3]:
        print(f"  - {t[:100]}")

Train unique: 4805
Test unique:  2017
Overlap:      9

⚠️ 9 exact duplicates found!
Sample overlaps:
  - [PHONE] এই নাম্বারে কল করুন, জরুরি।
  - Send bKash to this number [PHONE] .
  - [PHONE] ei number e call korun.


In [6]:
# ============================================================
# REMOVE 9 EXACT DUPLICATES FROM TEST SET
# ============================================================
import pandas as pd

# Load
train_df = pd.read_csv("data/processed/train_hard2.csv")
test_df = pd.read_csv("data/processed/test_hard2.csv")

print(f"Before cleaning:")
print(f"  Train: {len(train_df)}")
print(f"  Test:  {len(test_df)}")

# Find overlaps
train_texts = set(train_df['text_clean'].astype(str))
test_df['is_duplicate'] = test_df['text_clean'].astype(str).isin(train_texts)

n_duplicates = test_df['is_duplicate'].sum()
print(f"\n⚠️ Found {n_duplicates} exact duplicates in test")

# Show them
duplicates = test_df[test_df['is_duplicate']]
print(f"\nDuplicate samples:")
for i, (_, row) in enumerate(duplicates.iterrows(), 1):
    print(f"  {i}. [{row['label']}] {row['text_clean'][:100]}")

# Remove duplicates
test_df_clean = test_df[~test_df['is_duplicate']].drop('is_duplicate', axis=1).reset_index(drop=True)

print(f"\nAfter cleaning:")
print(f"  Test: {len(test_df_clean)}")

# Verify — no overlap now
test_texts_clean = set(test_df_clean['text_clean'].astype(str))
final_overlap = train_texts & test_texts_clean
print(f"\n✅ Final overlap: {len(final_overlap)}")

# Save
test_df_clean.to_csv("data/processed/test_hard2_clean.csv", index=False)
print(f"\n✅ Saved: data/processed/test_hard2_clean.csv")

Before cleaning:
  Train: 4898
  Test:  2107

⚠️ Found 9 exact duplicates in test

Duplicate samples:
  1. [normal] [PHONE] number-এ bKash করুন।
  2. [normal] [PHONE] এই নাম্বারে কল করুন, জরুরি।
  3. [normal] [PHONE] number e call korun.
  4. [normal] [PHONE] এই নাম্বারে কল করুন।
  5. [normal] Send bKash to this number [PHONE] .
  6. [normal] [PHONE] এই নাম্বারে কল করুন, আমার টাকা দরকার।
  7. [normal] [PHONE] ei number e call korun.
  8. [normal] [PHONE] ei number e druto taka pathao.
  9. [normal] [PHONE] নাম্বারে বিকাশ করুন।

After cleaning:
  Test: 2098

✅ Final overlap: 0

✅ Saved: data/processed/test_hard2_clean.csv


In [7]:
# ============================================================
# CHECK B2 FOR DUPLICATES
# ============================================================
train_b2 = pd.read_csv("data/processed/train_hard3.csv")
test_b2 = pd.read_csv("data/processed/test_hard3.csv")

train_b2_texts = set(train_b2['text_clean'].astype(str))
test_b2['is_dup'] = test_b2['text_clean'].astype(str).isin(train_b2_texts)

print(f"B2 overlap: {test_b2['is_dup'].sum()}")

if test_b2['is_dup'].sum() > 0:
    test_b2_clean = test_b2[~test_b2['is_dup']].drop('is_dup', axis=1).reset_index(drop=True)
    test_b2_clean.to_csv("data/processed/test_hard3_clean.csv", index=False)
    print(f"✅ Saved: test_hard3_clean.csv ({len(test_b2_clean)} samples)")

B2 overlap: 9
✅ Saved: test_hard3_clean.csv (2834 samples)


In [8]:
# ============================================================
# RE-EVALUATE B1 ON CLEAN TEST SET
# ============================================================
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import PeftModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ---------- Load clean test ----------
test_df_clean = pd.read_csv("data/processed/test_hard2_clean.csv")
print(f"Clean B1 test size: {len(test_df_clean)}")

# ---------- Load model ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
base_model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
model = PeftModel.from_pretrained(base_model, "results/models/lora_b1_final")
model.to(device).eval()

# ---------- Dataset ----------
class SMSDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len
        )

labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}

texts = test_df_clean['text_clean'].astype(str).tolist()
y_true = np.array([label2id[l] for l in test_df_clean['label']])

ds = SMSDataset(texts, tokenizer)
loader = DataLoader(ds, batch_size=32, collate_fn=DataCollatorWithPadding(tokenizer))

# ---------- Predict ----------
all_preds = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = torch.argmax(model(**batch).logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())

y_pred = np.array(all_preds)

# ---------- Metrics ----------
f1_new = f1_score(y_true, y_pred, average='macro')
acc_new = accuracy_score(y_true, y_pred)

smish_idx = label2id['smish']
smish_mask = y_true == smish_idx
recall_new = (y_pred[smish_mask] == smish_idx).mean()

print("\n" + "=" * 60)
print("B1 CLEAN TEST RESULTS")
print("=" * 60)
print(f"Test size:      {len(test_df_clean)}  (was 2107)")
print(f"Macro F1:       {f1_new:.4f}  (was 0.7463)")
print(f"Accuracy:       {acc_new:.4f}  (was 0.7451)")
print(f"Smish Recall:   {recall_new:.4f}  (was 0.5888)")
print("=" * 60)

# Save
b1_clean = {
    "test_size": len(test_df_clean),
    "removed_duplicates": 2107 - len(test_df_clean),
    "macro_f1": float(f1_new),
    "accuracy": float(acc_new),
    "smish_recall": float(recall_new),
    "previous_f1": 0.7463,
    "previous_recall": 0.5888,
}
with open("results/analysis/b1_clean_results.json", "w") as f:
    json.dump(b1_clean, f, indent=2)

print("\n✅ Saved: results/analysis/b1_clean_results.json")

Clean B1 test size: 2098


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



B1 CLEAN TEST RESULTS
Test size:      2098  (was 2107)
Macro F1:       0.7452  (was 0.7463)
Accuracy:       0.7440  (was 0.7451)
Smish Recall:   0.5888  (was 0.5888)

✅ Saved: results/analysis/b1_clean_results.json


In [10]:
# ============================================================
# RE-EVALUATE B2 ON CLEAN TEST SET
# ============================================================
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import PeftModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ---------- Load clean B2 test ----------
test_b2_clean = pd.read_csv("data/processed/test_hard3_clean.csv")
print(f"Clean B2 test size: {len(test_b2_clean)}")

# ---------- Load B2 model ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
base_model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
model_b2 = PeftModel.from_pretrained(base_model, "results/models/lora_b2_final")
model_b2.to(device).eval()

# ---------- Dataset ----------
class SMSDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len
        )

labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}

texts = test_b2_clean['text_clean'].astype(str).tolist()
y_true = np.array([label2id[l] for l in test_b2_clean['label']])

ds = SMSDataset(texts, tokenizer)
loader = DataLoader(ds, batch_size=32, collate_fn=DataCollatorWithPadding(tokenizer))

# ---------- Predict ----------
all_preds = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = torch.argmax(model_b2(**batch).logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())

y_pred = np.array(all_preds)

# ---------- Metrics ----------
f1_b2 = f1_score(y_true, y_pred, average='macro')
acc_b2 = accuracy_score(y_true, y_pred)

smish_idx = label2id['smish']
smish_mask = y_true == smish_idx
recall_b2 = (y_pred[smish_mask] == smish_idx).mean()

print("\n" + "=" * 60)
print("B2 CLEAN TEST RESULTS")
print("=" * 60)
print(f"Test size:      {len(test_b2_clean)}  (was 2843)")
print(f"Macro F1:       {f1_b2:.4f}")
print(f"Accuracy:       {acc_b2:.4f}")
print(f"Smish Recall:   {recall_b2:.4f}")
print("=" * 60)

# Save
b2_clean = {
    "test_size": len(test_b2_clean),
    "removed_duplicates": 2843 - len(test_b2_clean),
    "macro_f1": float(f1_b2),
    "accuracy": float(acc_b2),
    "smish_recall": float(recall_b2),
}
with open("results/analysis/b2_clean_results.json", "w") as f:
    json.dump(b2_clean, f, indent=2)

print("\n✅ Saved: results/analysis/b2_clean_results.json")

Clean B2 test size: 2834


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



B2 CLEAN TEST RESULTS
Test size:      2834  (was 2843)
Macro F1:       0.7596
Accuracy:       0.7865
Smish Recall:   0.7089

✅ Saved: results/analysis/b2_clean_results.json


In [11]:
# ============================================================
# BOOTSTRAP CI ON CLEAN B1 TEST SET
# ============================================================
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import PeftModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ---------- Load clean B1 test ----------
test_df_clean = pd.read_csv("data/processed/test_hard2_clean.csv")

# ---------- Load model ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
base_model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
model = PeftModel.from_pretrained(base_model, "results/models/lora_b1_final")
model.to(device).eval()

# ---------- Dataset ----------
class SMSDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len
        )

labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}

texts = test_df_clean['text_clean'].astype(str).tolist()
y_true = np.array([label2id[l] for l in test_df_clean['label']])

ds = SMSDataset(texts, tokenizer)
loader = DataLoader(ds, batch_size=32, collate_fn=DataCollatorWithPadding(tokenizer))

# ---------- Predict ----------
all_preds = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = torch.argmax(model(**batch).logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())

y_pred = np.array(all_preds)

# ---------- Bootstrap ----------
def bootstrap_ci(y_true, y_pred, metric_fn, n_iterations=1000, seed=42):
    scores = []
    np.random.seed(seed)
    for _ in range(n_iterations):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

print("=" * 60)
print("BOOTSTRAP 95% CI (Clean B1 Test Set, n=2098)")
print("=" * 60)

# Macro F1
mean, lo, hi = bootstrap_ci(y_true, y_pred, 
    lambda t, p: f1_score(t, p, average='macro'))
print(f"Macro F1:        {mean:.4f} [{lo:.4f}, {hi:.4f}]")

# Smish Recall
smish_idx = label2id['smish']
mean, lo, hi = bootstrap_ci(y_true, y_pred,
    lambda t, p: (p[t==smish_idx]==smish_idx).mean())
print(f"Smish Recall:    {mean:.4f} [{lo:.4f}, {hi:.4f}]")

# Smish Precision
mean, lo, hi = bootstrap_ci(y_true, y_pred,
    lambda t, p: (t[p==smish_idx]==smish_idx).mean() if (p==smish_idx).sum()>0 else 0)
print(f"Smish Precision: {mean:.4f} [{lo:.4f}, {hi:.4f}]")

# Accuracy
mean, lo, hi = bootstrap_ci(y_true, y_pred, accuracy_score)
print(f"Accuracy:        {mean:.4f} [{lo:.4f}, {hi:.4f}]")

print("=" * 60)

# Save
import json
ci_results = {
    "test_size": len(test_df_clean),
    "macro_f1": {"mean": float(mean), "ci_low": float(lo), "ci_high": float(hi)},
}
with open("results/analysis/bootstrap_ci_clean.json", "w") as f:
    json.dump(ci_results, f, indent=2)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BOOTSTRAP 95% CI (Clean B1 Test Set, n=2098)
Macro F1:        0.7455 [0.7257, 0.7641]
Smish Recall:    0.5891 [0.5613, 0.6167]
Smish Precision: 0.9868 [0.9781, 0.9935]
Accuracy:        0.7444 [0.7245, 0.7631]


In [12]:
# ============================================================
# CELL 5: AUGMENTATION ABLATION
# Train WITHOUT augmentation to measure its effect
# ============================================================
import gc
import json
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType

gc.collect()
torch.cuda.empty_cache()

# ---------- Load data (WITHOUT augmentation) ----------
train_df = pd.read_csv("data/processed/train_hard2.csv")
test_df = pd.read_csv("data/processed/test_hard2_clean.csv")  # CLEAN test

print(f"Train (no aug): {len(train_df)}")
print(f"Test (clean):   {len(test_df)}")

# ---------- Setup ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# ---------- Dataset ----------
class SMSDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_len=128):
        self.texts = df['text_clean'].astype(str).tolist()
        self.labels = [label2id[l] for l in df['label']]
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len
        )
        enc['labels'] = self.labels[idx]
        return enc

train_ds = SMSDataset(train_df, tokenizer, label2id)
test_ds = SMSDataset(test_df, tokenizer, label2id)

# ---------- Class weights ----------
lc = Counter(train_df['label'].map(label2id))
tot = sum(lc.values())
cw = {i: tot / (len(labels) * lc[i]) for i in range(len(labels))}
print(f"Class weights: {cw}")

# ---------- Model (LoRA) ----------
base_model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base", num_labels=3,
    id2label=id2label, label2id=label2id
)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, lora_alpha=32, lora_dropout=0.2, bias="none",
    target_modules=["query", "key", "value", "dense"]
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# ---------- Weighted Trainer ----------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        y = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        w = torch.tensor(
            [self.class_weights[i] for i in range(len(self.class_weights))],
            device=logits.device, dtype=torch.float32
        )
        loss = torch.nn.CrossEntropyLoss(weight=w)(logits, y)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(y_true, y_pred, average='macro'),
        "accuracy": accuracy_score(y_true, y_pred)
    }

# ---------- Training ----------
training_args = TrainingArguments(
    output_dir="results/models/lora_noaug",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.05,
    warmup_steps=0.2,
    max_grad_norm=0.5,
    label_smoothing_factor=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    optim="adamw_torch_fused",
    seed=42
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    class_weights=cw
)

print("\n🚀 Training WITHOUT augmentation...")
trainer.train()
print("✅ Training complete")

# ---------- Evaluate ----------
results = trainer.evaluate(test_ds, metric_key_prefix="test")
f1_noaug = results['test_macro_f1']

preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
smish_idx = label2id['smish']
mask = y_true == smish_idx
recall_noaug = (y_pred[mask] == smish_idx).mean()

print("\n" + "=" * 60)
print("AUGMENTATION ABLATION RESULT")
print("=" * 60)
print(f"WITHOUT augmentation:")
print(f"  Test F1:        {f1_noaug:.4f}")
print(f"  Smish Recall:   {recall_noaug:.4f}")
print(f"\nWITH augmentation (main result):")
print(f"  Test F1:        0.7452")
print(f"  Smish Recall:   0.5888")
print(f"\nImprovement:")
print(f"  ΔF1:            {0.7452 - f1_noaug:+.4f}")
print(f"  ΔRecall:        {0.5888 - recall_noaug:+.4f}")
print("=" * 60)

# ---------- Save ----------
ablation = {
    "without_augmentation": {
        "test_f1": float(f1_noaug),
        "smish_recall": float(recall_noaug),
    },
    "with_augmentation": {
        "test_f1": 0.7452,
        "smish_recall": 0.5888,
    },
    "improvement": {
        "f1": float(0.7452 - f1_noaug),
        "recall": float(0.5888 - recall_noaug),
    }
}
with open("results/analysis/ablation_augmentation.json", "w") as f:
    json.dump(ablation, f, indent=2)

print("\n✅ Saved: results/analysis/ablation_augmentation.json")

Train (no aug): 4898
Test (clean):   2098
Class weights: {0: 0.8204355108877722, 1: 1.1952171791117618, 2: 1.0587980977086036}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,247,107 || all params: 281,293,062 || trainable%: 1.1544

🚀 Training WITHOUT augmentation...


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.620287,0.560522,0.739066,0.749285
2,0.145284,2.039318,0.550617,0.540038
3,0.132276,1.972751,0.593359,0.585796


✅ Training complete


Training Loss,Validation Loss,Epoch,Macro F1,Accuracy
0.132276,0.560522,3,0.739066,0.749285



AUGMENTATION ABLATION RESULT
WITHOUT augmentation:
  Test F1:        0.7391
  Smish Recall:   0.6330

WITH augmentation (main result):
  Test F1:        0.7452
  Smish Recall:   0.5888

Improvement:
  ΔF1:            +0.0061
  ΔRecall:        -0.0442

✅ Saved: results/analysis/ablation_augmentation.json
